# Inferência com YOLOv8 — Detecção de Agentes do Valorant

**Aluno:** Gustavo Henrique Femiano  
**Projeto:** Valorant AI — Detector customizado com YOLOv8

Este notebook apresenta a etapa de **inferência** do projeto. O treinamento do modelo já foi realizado anteriormente; portanto, aqui o objetivo é apenas carregar o arquivo treinado (`modelo_valorant.pt`) e testar sua capacidade de detectar agentes/personagens do Valorant em imagens e vídeos.

A entrega foi organizada para que o projeto possa ser executado em outro computador sem a necessidade de enviar o dataset completo. Para isso, basta manter o modelo treinado na pasta correta e executar as células deste notebook.


## 1. Visão geral do projeto

O projeto utiliza o modelo **YOLOv8**, da biblioteca Ultralytics, aplicado a um dataset customizado de imagens do Valorant. A proposta segue o conteúdo trabalhado nas aulas de treinamento customizado e inferência com YOLO: primeiro foi feito o treinamento com imagens anotadas; depois, o melhor modelo gerado (`best.pt`) foi separado para ser usado em testes práticos.

Nesta versão de entrega, o notebook não realiza novo treinamento. Ele executa apenas a inferência, isto é, usa o modelo já treinado para identificar objetos/classes em novos arquivos de entrada.

### Configuração usada no treinamento

O modelo foi treinado a partir do `yolov8n.pt`, que é a versão Nano do YOLOv8. A configuração principal utilizada no treinamento foi:

```text
Modelo base: yolov8n.pt
Épocas de treinamento: 30
Tamanho das imagens: 416
Batch size: 2
Nome do experimento: valorant_yolo
Arquivo final usado na inferência: models/modelo_valorant.pt
```

Uma época representa uma passagem completa do modelo pelo conjunto de treinamento. Neste projeto, o modelo estudou o dataset por **30 épocas**, ajustando seus pesos a cada rodada para melhorar as detecções.


## 2. Estrutura esperada do projeto

A estrutura abaixo mostra como os arquivos devem estar organizados para que o professor consiga executar o notebook corretamente.

```text
valorant-ai/
├── notebooks/
│   └── 12-valorant-inferencia-modelo-treinado-entrega.ipynb
├── models/
│   └── modelo_valorant.pt
├── inputs/
│   ├── imagem_teste.jpg        # opcional: imagem para teste
│   └── teste.mp4               # opcional: vídeo para teste
├── outputs/                    # criada automaticamente para salvar resultados
├── pyproject.toml              # dependências do projeto com Poetry
├── poetry.lock                 # versões travadas das bibliotecas
├── requirements.txt            # alternativa para instalação com pip, se existir
└── .gitignore                  # define o que entra ou não no GitHub
```

### Função de cada pasta/arquivo

- **`notebooks/`**: contém os notebooks Jupyter do projeto.
- **`models/`**: armazena o modelo treinado. O arquivo principal deve se chamar `modelo_valorant.pt`.
- **`inputs/`**: guarda imagens e vídeos usados nos testes de inferência.
- **`outputs/`**: recebe os resultados gerados pelo YOLO, incluindo imagens/vídeos anotados e arquivos `.txt` com as detecções.
- **`pyproject.toml` e `poetry.lock`**: permitem instalar o ambiente do projeto de forma reprodutível com Poetry.
- **`.gitignore`**: evita enviar arquivos pesados desnecessários, como dataset completo, vídeos grandes, pasta `runs/` e ambiente virtual.

O dataset completo não é necessário para executar este notebook, pois o modelo já foi treinado previamente.


## 3. Importações e configuração inicial

Nesta etapa, são importadas as bibliotecas necessárias e são definidos os caminhos principais do projeto. O código identifica automaticamente se o notebook está sendo executado dentro da pasta `notebooks/` e ajusta a raiz do projeto.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import base64
import shutil
import subprocess

import cv2
from IPython.display import Image, HTML, display
from ultralytics import YOLO, settings

# Localiza automaticamente a raiz do projeto.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"
INPUTS_DIR = PROJECT_ROOT / "inputs"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Cria as pastas necessárias, caso elas ainda não existam.
MODELS_DIR.mkdir(exist_ok=True)
INPUTS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# Configura a pasta local onde o Ultralytics pode salvar resultados auxiliares.
settings.update({"runs_dir": str(PROJECT_ROOT / "runs")})

print("Raiz do projeto:", PROJECT_ROOT)
print("Pasta do modelo:", MODELS_DIR)
print("Pasta de entradas:", INPUTS_DIR)
print("Pasta de saídas:", OUTPUTS_DIR)


## 4. Verificação da estrutura local

A célula abaixo verifica se as principais pastas e arquivos estão presentes. Essa checagem ajuda a identificar rapidamente problemas de caminho antes de executar a inferência.


In [ ]:
EXPECTED_ITEMS = {
    "Pasta notebooks": PROJECT_ROOT / "notebooks",
    "Pasta models": MODELS_DIR,
    "Modelo treinado": MODELS_DIR / "modelo_valorant.pt",
    "Pasta inputs": INPUTS_DIR,
    "Pasta outputs": OUTPUTS_DIR,
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "poetry.lock": PROJECT_ROOT / "poetry.lock",
}

print("Verificação da estrutura do projeto")
print("-" * 70)

for description, path in EXPECTED_ITEMS.items():
    status = "OK" if path.exists() else "NÃO ENCONTRADO"
    print(f"{description:<20} | {status:<15} | {path}")


## 5. Carregamento do modelo treinado

O arquivo esperado é `models/modelo_valorant.pt`. Esse arquivo é uma cópia renomeada do `best.pt`, que é o melhor checkpoint salvo pelo YOLO durante o treinamento.

A célula também considera alguns caminhos alternativos, caso o notebook seja aberto diretamente pela pasta `notebooks/`.


In [ ]:
possible_model_paths = [
    MODELS_DIR / "modelo_valorant.pt",
    PROJECT_ROOT / "models" / "modelo_valorant.pt",
    Path("models/modelo_valorant.pt"),
    Path("../models/modelo_valorant.pt"),
]

MODEL_PATH = None

for path in possible_model_paths:
    if path.exists():
        MODEL_PATH = path.resolve()
        break

if MODEL_PATH is None:
    raise FileNotFoundError(
        "Modelo treinado não encontrado.\n"
        "Coloque o arquivo em: models/modelo_valorant.pt\n"
        "Esse arquivo deve ser uma cópia do best.pt gerado no treinamento."
    )

print("Modelo encontrado em:", MODEL_PATH)

model = YOLO(str(MODEL_PATH))

print("Modelo carregado com sucesso.")
print("Classes conhecidas pelo modelo:")

for class_id, class_name in model.names.items():
    print(f"{class_id}: {class_name}")

## 6. Informações do treinamento armazenadas no checkpoint

O arquivo `.pt` pode guardar metadados do treinamento, como quantidade de épocas e argumentos usados. A célula abaixo tenta ler essas informações. Caso algum dado não esteja disponível no checkpoint, é mantida a configuração documentada do treinamento feito neste projeto.


In [ ]:
# Configuração documentada do treinamento realizado no projeto.
TRAINING_INFO = {
    "Modelo base": "yolov8n.pt",
    "Épocas de treinamento": 30,
    "Tamanho da imagem": 416,
    "Batch size": 2,
    "Nome do experimento": "valorant_yolo",
}

# Tentativa de leitura de metadados diretamente do checkpoint.
checkpoint = getattr(model, "ckpt", None)
train_args = {}
checkpoint_epoch = None

if isinstance(checkpoint, dict):
    train_args = checkpoint.get("train_args") or checkpoint.get("args") or {}
    checkpoint_epoch = checkpoint.get("epoch")

print("Resumo do treinamento")
print("-" * 70)

for key, value in TRAINING_INFO.items():
    print(f"{key:<25}: {value}")

if train_args:
    print("\nArgumentos encontrados no checkpoint:")
    for key in ["model", "data", "epochs", "imgsz", "batch", "name", "device"]:
        if key in train_args:
            print(f"{key:<10}: {train_args[key]}")

if checkpoint_epoch is not None:
    print("\nÚltima época registrada no checkpoint:", checkpoint_epoch)

## 7. Teste em imagem

Esta etapa realiza inferência em uma imagem de teste. Para usar, coloque uma imagem na pasta `inputs/` com um dos nomes abaixo:

```text
inputs/imagem_teste.jpg
inputs/imagem_teste.png
inputs/teste.jpg
inputs/teste.png
```

O resultado anotado será salvo na pasta `outputs/imagem_teste/` e uma cópia será exibida no próprio notebook.


In [ ]:
CONFIDENCE = 0.25

IMAGE_CANDIDATES = [
    INPUTS_DIR / "imagem_teste.jpg",
    INPUTS_DIR / "imagem_teste.png",
    INPUTS_DIR / "teste.jpg",
    INPUTS_DIR / "teste.png",
]

# Se os nomes principais não forem encontrados, procura a primeira imagem comum dentro de inputs/.
for extension in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
    IMAGE_CANDIDATES.extend(sorted(INPUTS_DIR.glob(extension)))

IMAGE_PATH = next((path for path in IMAGE_CANDIDATES if path.exists()), None)

if IMAGE_PATH is None:
    print("Nenhuma imagem foi encontrada para teste.")
    print("Coloque uma imagem em inputs/imagem_teste.jpg ou altere os nomes da célula.")
else:
    print("Imagem selecionada:", IMAGE_PATH)

    image_results = model.predict(
        source=str(IMAGE_PATH),
        conf=CONFIDENCE,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(OUTPUTS_DIR),
        name="imagem_teste",
        exist_ok=True,
        verbose=False,
    )

    annotated = image_results[0].plot()
    annotated_path = OUTPUTS_DIR / "imagem_teste_anotada.jpg"
    cv2.imwrite(str(annotated_path), annotated)

    print("Resultado salvo em:", OUTPUTS_DIR / "imagem_teste")
    display(Image(filename=str(annotated_path), width=900))


## 8. Teste em lote de imagens

A célula abaixo analisa todas as imagens encontradas na pasta `inputs/`. Essa etapa é útil para testar várias imagens de uma só vez e verificar se o modelo mantém um comportamento consistente em exemplos diferentes.


In [ ]:
image_files = []
for extension in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
    image_files.extend(sorted(INPUTS_DIR.glob(extension)))

if not image_files:
    print("Nenhuma imagem encontrada em inputs/.")
else:
    print(f"Foram encontradas {len(image_files)} imagem(ns) em inputs/.")

    batch_results = model.predict(
        source=str(INPUTS_DIR),
        conf=CONFIDENCE,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(OUTPUTS_DIR),
        name="lote_imagens",
        exist_ok=True,
        verbose=False,
    )

    result_dir = OUTPUTS_DIR / "lote_imagens"
    print("Resultados salvos em:", result_dir)

    result_images = []
    for extension in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        result_images.extend(sorted(result_dir.glob(extension)))

    for img in result_images[:5]:
        print("Exibindo:", img.name)
        display(Image(filename=str(img), width=850))


## 9. Teste em vídeo

Esta etapa executa a inferência em um vídeo. O vídeo de entrada deve estar dentro da pasta `inputs/`. Por padrão, o notebook procura por:

```text
inputs/teste.mp4
```

Se o YOLO salvar o resultado em `.avi`, o código tenta converter automaticamente para `.mp4`, pois esse formato costuma ser mais compatível com a exibição dentro do Jupyter/VS Code.


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64
import shutil
import subprocess
import cv2

CONFIDENCE = 0.25

VIDEO_NAME = "teste.mp4"
VIDEO_PATH = INPUTS_DIR / VIDEO_NAME


def encontrar_video_saida(pasta_saida: Path):
    output_videos = []

    for extension in ("*.mp4", "*.avi", "*.mov", "*.mkv", "*.webm"):
        output_videos.extend(pasta_saida.glob(extension))

    if not output_videos:
        return None

    return max(output_videos, key=lambda path: path.stat().st_mtime)


def obter_ffmpeg():
    ffmpeg_path = shutil.which("ffmpeg")

    if ffmpeg_path:
        return ffmpeg_path

    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None


def converter_para_mp4_h264(video_entrada: Path) -> Path:
    video_saida = video_entrada.with_name(video_entrada.stem + "_convertido.mp4")
    ffmpeg_path = obter_ffmpeg()

    if ffmpeg_path is not None:
        comando = [
            ffmpeg_path,
            "-y",
            "-i", str(video_entrada),
            "-vcodec", "libx264",
            "-pix_fmt", "yuv420p",
            "-preset", "fast",
            "-crf", "23",
            str(video_saida),
        ]

        subprocess.run(comando, check=True)
        return video_saida

    print("FFmpeg não encontrado. Tentando converter com OpenCV...")

    cap = cv2.VideoCapture(str(video_entrada))

    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir o vídeo: {video_entrada}")

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(video_saida), fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        writer.write(frame)

    cap.release()
    writer.release()

    return video_saida


def exibir_video_mp4(video_path: Path, width=900):
    video_bytes = video_path.read_bytes()
    video_base64 = base64.b64encode(video_bytes).decode("utf-8")

    html = f"""
    <video width="{width}" controls>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
        Seu navegador não conseguiu exibir o vídeo.
    </video>
    """

    display(HTML(html))


if not VIDEO_PATH.exists():
    print("Vídeo não encontrado:", VIDEO_PATH)
    print("Coloque um vídeo em inputs/teste.mp4 ou altere a variável VIDEO_NAME.")
else:
    print("Vídeo selecionado:", VIDEO_PATH)

    video_results = model.predict(
        source=str(VIDEO_PATH),
        conf=CONFIDENCE,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(OUTPUTS_DIR),
        name="video_teste",
        exist_ok=True,
        verbose=False,
    )

    video_result_dir = OUTPUTS_DIR / "video_teste"
    print("Resultado salvo em:", video_result_dir)

    output_video = encontrar_video_saida(video_result_dir)

    if output_video is None:
        print("O vídeo foi processado, mas nenhum arquivo de vídeo foi encontrado na pasta de saída.")
    else:
        print("Vídeo gerado pelo YOLO:", output_video)

        try:
            video_mp4 = converter_para_mp4_h264(output_video)

            print("Vídeo final para exibição:", video_mp4)
            print("Se o vídeo não aparecer abaixo, abra manualmente o arquivo acima.")

            exibir_video_mp4(video_mp4, width=900)

        except Exception as erro:
            print("Não foi possível converter/exibir o vídeo dentro do notebook.")
            print("Erro:", erro)
            print("Abra manualmente o arquivo gerado em:", output_video)

## 10. Webcam opcional

A inferência também pode ser feita pela webcam do computador. Esta etapa está desativada por padrão para evitar abrir uma janela automaticamente durante a correção.

Para testar, altere `USAR_WEBCAM` para `True` e execute a célula.


In [ ]:
USAR_WEBCAM = False

if USAR_WEBCAM:
    print("Abrindo webcam. Feche a janela do YOLO/OpenCV para encerrar.")
    model.predict(source=0, conf=CONFIDENCE, show=True)
else:
    print("Webcam desativada. Para ativar, altere USAR_WEBCAM para True.")


## 11. Resumo das detecções geradas

Quando a inferência é executada com `save_txt=True` e `save_conf=True`, o YOLO cria arquivos `.txt` com as detecções. A célula abaixo lê esses arquivos e gera um resumo simples com quantidade de detecções por classe e confiança média.

Por padrão, o resumo considera os resultados do vídeo. Para analisar outro teste, altere a variável `RESUMO_DIR`.


In [ ]:
# Opções comuns:
# RESUMO_DIR = OUTPUTS_DIR / "video_teste"
# RESUMO_DIR = OUTPUTS_DIR / "imagem_teste"
# RESUMO_DIR = OUTPUTS_DIR / "lote_imagens"
RESUMO_DIR = OUTPUTS_DIR / "video_teste"
LABELS_DIR = RESUMO_DIR / "labels"

if not LABELS_DIR.exists():
    print("Pasta de labels não encontrada:", LABELS_DIR)
    print("Execute primeiro uma inferência com save_txt=True.")
else:
    label_files = sorted(LABELS_DIR.glob("*.txt"))
    contador = Counter()
    confiancas = defaultdict(list)

    for label_file in label_files:
        with open(label_file, "r", encoding="utf-8") as arquivo:
            for linha in arquivo:
                partes = linha.strip().split()
                if not partes:
                    continue

                class_id = partes[0]
                contador[class_id] += 1

                if len(partes) >= 6:
                    try:
                        confiancas[class_id].append(float(partes[-1]))
                    except ValueError:
                        pass

    if not contador:
        print("Nenhuma detecção foi encontrada.")
        print("É possível reduzir o valor de CONFIDENCE ou testar outro arquivo.")
    else:
        print("Resumo das detecções")
        print("-" * 72)
        print(f"{'ID':<8} {'Classe':<30} {'Quantidade':<12} {'Conf. média'}")
        print("-" * 72)

        for class_id, quantidade in contador.most_common():
            class_name = model.names.get(int(class_id), class_id) if str(class_id).isdigit() else class_id
            if confiancas[class_id]:
                media = sum(confiancas[class_id]) / len(confiancas[class_id])
                media_txt = f"{media:.3f}"
            else:
                media_txt = "-"

            print(f"{class_id:<8} {str(class_name):<30} {quantidade:<12} {media_txt}")

        print("-" * 72)
        print("Arquivos de label analisados:", len(label_files))


## 12. Considerações finais

Este notebook representa a etapa final do projeto: usar o modelo treinado para realizar detecções em novos arquivos. A pasta do dataset não precisa ser enviada para o GitHub porque ela foi necessária apenas durante o treinamento.

Para executar o projeto em outra máquina, é necessário manter pelo menos os seguintes arquivos:

```text
notebooks/12-valorant-inferencia-modelo-treinado-entrega.ipynb
models/modelo_valorant.pt
pyproject.toml
poetry.lock
README.md
```

Com Poetry, a execução pode ser feita com:

```bash
poetry install
poetry run jupyter notebook
```

Depois disso, basta abrir este notebook, selecionar o kernel do projeto e executar as células na ordem.
